**!!! NOTEBOOK FOR USAGE REFERENCE !!!**

This notebook is not intended to provide actual code; Instead, it is just a "getting started", something you can take as inspiration when using this testbed.

For developers: please do not commit the outputs to the library. Always clear all outputs before saving.

----

Downloads, enrich, filter, save splits

Generates metadata_filtered

General notebook structure:
1. Inicial set: ~300 entities to be partially enriched
2. Enriched set: fully enriched and verified, including manual checks/inputs. Manually remove mismaches name<->pantheon.
3. Filtered set: filtered 100 entities based on representativenss balancing
4. Entities appear exactly in this order in the table representations.
5. Images should only be saved for the entities in the final enriched set


In [ ]:
import os
import sys
import json
import dotenv
import pandas as pd
import torch.utils.checkpoint

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_colwidth', 30)

sys.path.append('../TRDP-unlearning')
dotenv.load_dotenv('../TRDP-unlearning/SD_lora_distil/.env')
os.environ['WANDB_DISABLED'] = "true"
assert len(os.getenv('HF_TOKEN'))>0
#!huggingface-cli login --token ${HF_TOKEN}

from vision_unlearning.datasets import download_dataset_akc, akc_find_closest_match, count_classes_dataset_taras_breeds, download_dataset_taras_breeds, split_dataset_taras_breeds, balanced_subsample_lib
from vision_unlearning.datasets.testbed import calculate_similarity_clip, plot_heatmap
from vision_unlearning.utils.logger import get_logger, setup_loggers


logger = get_logger('testbed')
setup_loggers()

In [ ]:
dataset_base_path = 'assets/datasets/taras_breeds'
dataset_base_path_filtered = 'assets/datasets/taras_breeds_splits_filtered'
cache_folder_taras = "assets/datasets/temp_taras_unconverted"

# Restart from step:
#!rm assets/metadata_breeds_1_enriched_but_not_filtered.json && rm assets/metadata_breeds_2_enriched_filtered.json && rm -r {dataset_base_path_filtered}
#!rm assets/metadata_breeds_2_enriched_filtered.json && rm -r {dataset_base_path_filtered}
#!rm -r {dataset_base_path_filtered}

#!rm -rf {dataset_base_path}
#!rm -rf {cache_folder_taras}
#!rm -rf "assets/datasets/akc-data-latest.csv"
#!rm "assets/similarity_clip_breeds.json"


In [ ]:
##############################
# Step 1: prepare datasets
##############################
# Main image dataset's attributes: Taras Breeds
# https://github.com/AtharvaTaras/Dog-Breeds-Dataset
download_dataset_taras_breeds(dataset_base_path, cache_folder_taras)
sorted_counts = count_classes_dataset_taras_breeds(dataset_base_path)

df = pd.read_csv(os.path.join(dataset_base_path, 'FCI Breeds.csv'), index_col='id')
df['name'] = df['name'].str.upper().str.replace('DOG', '').str.strip()
df.head()

In [ ]:
# Prepare attribute dataset AKC
# https://github.com/tmfilho/akcdata
# Scrapped from https://www.akc.org/

# There are a few examples that could still be matches, like:
'''
print(list(filter(lambda b: b['name']=='manchester terrier dog', metadata)))
name = 'manchester'
df_akc_pawsome[(df_akc_pawsome['name_akc'].str.upper().str.contains(name.upper())) | (df_akc_pawsome['Other Names'].str.upper().str.contains(name.upper()))][['name_akc', 'Other Names', 'group_akc']]

print(list(filter(lambda b: b['name']=='kishu dog', metadata)))
name = 'kishu'
df_akc_pawsome[(df_akc_pawsome['name_akc'].str.upper().str.contains(name.upper())) | (df_akc_pawsome['Other Names'].str.upper().str.contains(name.upper()))][['name_akc', 'Other Names', 'group_akc']]
''';

#!rm assets/datasets/akc-data-latest.csv
df_akc = download_dataset_akc(output_path = "assets/datasets/akc-data-latest.csv")
df_akc.rename({'Unnamed: 0': 'name_akc', 'group': 'group_akc'}, axis=1, inplace=True)
df_akc.head(5)

In [ ]:
# Prepare pawsomeauthority dog breeds dataset
# https://pawsomeauthority.com/dog-breeds/dataset/
# They have only 60 classes, but nice attributes and list of name synonyms
assert os.path.exists('assets/datasets/dog_breeds_pawsomeauthority.csv'), 'This dataset must be downloaded manually'
df_pawsome = pd.read_csv('assets/datasets/dog_breeds_pawsomeauthority.csv')
df_pawsome.rename({'Breed Name': 'name_pawsome', 'Group': 'group_pawsome'}, axis=1, inplace=True)
df_pawsome.head(5)

In [ ]:
# Merge Pawsome and AKC
df_pawsome.set_index('name_pawsome', inplace=True)
df_akc_pawsome = df_akc.join(df_pawsome, on='name_akc', how='left', rsuffix='Pawsome ')
print(f"From the original {df_pawsome.shape[0]} pawsome rows, {(~df_akc_pawsome['group_pawsome'].isna()).sum()} matches were found")
assert df_akc.shape[0] == df_akc_pawsome.shape[0]
#df_akc_pawsome.info()

In [ ]:
##############################
# Step 2: attribute inference
##############################
# How much intersection is there between AKC and Taras?
# 22 in total and 10 among the filtered ones, so we need a more complicated matching
'''
print(len(sorted_counts))
labels_taras = [l[0].lower().replace(' ', '_') for l in sorted_counts]
labels_taras += [l.replace('dog', '').strip() for l in labels_taras]
print(len(set(df_akc['name_akc'].str.lower().str.replace(' ', '_')) & set(labels_taras) ))
print(len(set(df_balanced['name_akc'].str.lower().str.replace(' ', '_')) & set(labels_taras) ))
''';

# How much intersection is there between Stanford Dogs Dataset and Taras?
# http://vision.stanford.edu/aditya86/ImageNetDogs/
# 74 in total and 36 among the filtered ones, so don't serve our purposes
'''
labels_standford= ['papillon', 'West_Highland_white_terrier', 'dingo', 'cairn', 'Brittany_spaniel', 'Greater_Swiss_Mountain_dog', 'Doberman', 'Old_English_sheepdog', 'Bernese_mountain_dog', 'bull_mastiff', 'Great_Dane', 'Border_terrier', 'Leonberg', 'Eskimo_dog', 'Pomeranian', 'standard_schnauzer', 'affenpinscher', 'toy_terrier', 'Irish_water_spaniel', 'Scotch_terrier', 'Pembroke', 'Norwegian_elkhound', 'bloodhound', 'German_shepherd', 'Lakeland_terrier', 'Brabancon_griffon', 'Chihuahua', 'German_short-haired_pointer', 'miniature_pinscher', 'French_bulldog', 'Australian_terrier', 'Boston_bull', 'malamute', 'keeshond', 'soft-coated_wheaten_terrier', 'cocker_spaniel', 'groenendael', 'Maltese_dog', 'borzoi', 'Shetland_sheepdog', 'malinois', 'Norwich_terrier', 'vizsla', 'beagle', 'Afghan_hound', 'Tibetan_mastiff', 'golden_retriever', 'curly-coated_retriever', 'Chesapeake_Bay_retriever', 'Airedale', 'Bedlington_terrier', 'flat-coated_retriever', 'Blenheim_spaniel', 'Irish_setter', 'Border_collie', 'English_foxhound', 'miniature_poodle', 'toy_poodle', 'Siberian_husky', 'pug', 'Rottweiler', 'Gordon_setter', 'Walker_hound', 'Irish_terrier', 'Cardigan', 'American_Staffordshire_terrier', 'Newfoundland', 'Rhodesian_ridgeback', 'Mexican_hairless', 'giant_schnauzer', 'wire-haired_fox_terrier', 'Welsh_springer_spaniel', 'Ibizan_hound', 'basenji', 'dhole', 'redbone', 'Staffordshire_bullterrier', 'EntleBucher', 'collie', 'Kerry_blue_terrier', 'Samoyed', 'Great_Pyrenees', 'Sussex_spaniel', 'komondor', 'English_setter', 'boxer', 'Appenzeller', 'Tibetan_terrier', 'schipperke', 'African_hunting_dog', 'kelpie', 'otterhound', 'Irish_wolfhound', 'silky_terrier', 'Saluki', 'miniature_schnauzer', 'Pekinese', 'basset', 'black-and-tan_coonhound', 'Bouvier_des_Flandres', 'Scottish_deerhound', 'whippet', 'Dandie_Dinmont', 'Japanese_spaniel', 'standard_poodle', 'Weimaraner', 'briard', 'bluetick', 'clumber', 'chow', 'Italian_greyhound', 'English_springer', 'Yorkshire_terrier', 'Saint_Bernard', 'Sealyham_terrier', 'Labrador_retriever', 'Shih-Tzu', 'Lhasa', 'Norfolk_terrier', 'kuvasz']
labels_standford = [l.lower().replace(' ', '_') for l in labels_standford]
print(len(labels_standford))
print(len(set(df_akc['name_akc'].str.lower().str.replace(' ', '_')) & set(labels_standford) ))
print(len(set(df_balanced['name_akc'].str.lower().str.replace(' ', '_')) & set(labels_standford) ))
''';


#!rm assets/metadata_breeds_1_enriched_but_not_filtered.json && rm assets/metadata_breeds_2_enriched_filtered.json && rm -r {dataset_base_path_filtered}
if os.path.exists('assets/metadata_breeds_1_enriched_but_not_filtered.json'):
    with open(f"assets/metadata_breeds_1_enriched_but_not_filtered.json", "r", encoding="utf-8") as f:
        metadata = json.load(f)
else:
    # Inicial set
    metadata = [{'name': name, 'dataset_n_original': n} for name, n in sorted_counts[:500]]

    # Attribute inference option 1:labels from Taras itself
    # This logic is so simple that could be a join, if metadata were a df
    for entity in metadata:
        row = df[df['name'] == entity['name'].upper().replace('DOG', '').strip()].iloc[0]
        entity['group'] = row['group'].lower().replace(' ', '_')
        entity['section'] = str(row['section']).lower().replace(' ', '_')
        entity['country'] = row['country'].lower().replace(' ', '_')

    # Attribute inference option 2: AKC
    logger.info('---------------- Enriching metadata with ALC')
    assert len(set(metadata[0].keys()) & set(df_akc_pawsome.columns)) == 0, 'There are attribute naming conflicts'
    for entity in metadata:
        name_akc = akc_find_closest_match(df_akc_pawsome, entity['name'], entity['group'])
        if name_akc is None:
            continue
        entity['name_akc'] = name_akc
        akc_row = df_akc_pawsome[df_akc_pawsome['name_akc']==entity['name_akc']].iloc[0]
        entity.update(akc_row.to_dict())

    # Attribute inference option 3: average value in some labeled dataset
    # TODO
    
    # Attribute inference option 4: data-driven with DeepFace

    # Save
    with open(f"assets/metadata_breeds_1_enriched_but_not_filtered.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

In [ ]:
##############################
# Step 3: filter
##############################
# Other attributes that could be interesting: shedding_category, energy_level_category, trainability_category, demeanor_category
#!rm assets/metadata_breeds_2_enriched_filtered.json && rm -r {dataset_base_path_filtered}
balanced_n = 100

if os.path.exists('assets/metadata_breeds_2_enriched_filtered.json'):
    logger.info('Reading existing filtered data')
    with open(f"assets/metadata_breeds_2_enriched_filtered.json", "r", encoding="utf-8") as f:
        metadata_filtered = json.load(f)
else:
    logger.info('Filtering')
    with open(f"assets/metadata_breeds_1_enriched_but_not_filtered.json", "r", encoding="utf-8") as f:
        metadata = json.load(f)

    df_enriched = pd.DataFrame(metadata)
    print(f"Rows in unfiltered dataset: {df_enriched.shape[0]}")
    df_enriched.dropna(subset=['grooming_frequency_value', 'group_akc'], axis=0, inplace=True)
    print(f"Rows after dropping rows that could not be enriched: {df_enriched.shape[0]}")
    df_enriched['grooming_frequency_category_binary'] = df_enriched['grooming_frequency_value'] > df_enriched['grooming_frequency_value'].median() 
    
    print('BEFORE BALANCING')
    print(df_enriched.value_counts('grooming_frequency_category_binary'))
    print('-'*50)
    print(df_enriched.value_counts('group'))
    print('-'*50)
    print(df_enriched.value_counts(['grooming_frequency_category_binary', 'group']))
    
    df_enriched = df_enriched[df_enriched['group']!='sighthounds']  # only 7
    df_enriched = df_enriched[df_enriched['group']!='dachshunds']  # only 1
    df_enriched = df_enriched[df_enriched['group']!='pointing_dogs'] # Pointing dogs that require grooming: 2
    df_enriched = df_enriched[df_enriched['group']!='scent_hounds_and_related_breeds'] # scent_hounds that require grooming: 2
    df_balanced = balanced_subsample_lib(df_enriched, group_cols=['grooming_frequency_category_binary', 'group'], target=balanced_n, priority_col='popularity')
    print('-'*50)
    print('-')
    print('-'*50)
    print('BALANCED')
    
    print(df_balanced.value_counts('grooming_frequency_category_binary'))
    print('-'*50)
    print(df_balanced.value_counts('group'))
    print('-'*50)
    print(df_balanced.value_counts(['grooming_frequency_category_binary', 'group']))
    
    metadata_filtered = df_balanced.to_dict(orient='records')
    with open(f"assets/metadata_breeds_2_enriched_filtered.json", "w", encoding="utf-8") as f:
        json.dump(metadata_filtered, f, indent=2)

assert type(metadata_filtered) == list
assert len(metadata_filtered) == balanced_n
assert sum([b['name']=='griffon bruxellois dog' for b in metadata_filtered]) == 1

In [ ]:
##############################
# Step 4: save splits
##############################
smallest_entity = min([entity['dataset_n_original'] for entity in metadata_filtered])
restrict_labels = [e['name'] for e in metadata_filtered]
logger.info(f"We have {balanced_n} identities, each one with {smallest_entity} images")


#!rm -r {dataset_base_path_filtered}

if not os.path.exists(dataset_base_path_filtered):
    for i, target in enumerate(restrict_labels):
        logger.info(f"Saving split dataset for entity {i}: {target}")
        dataset_forget_name = f"{dataset_base_path_filtered}/{target}/train_forget"
        dataset_retain_name = f"{dataset_base_path_filtered}/{target}/train_retain"
        class_to_number = split_dataset_taras_breeds(
            dataset_base_path,
            dataset_forget_name,
            dataset_retain_name,
            target,
            forget_max_img = smallest_entity,
            retain_max_img_per_class= smallest_entity,
            restrict_labels = restrict_labels,
        )
        assert sum([v>0 for v in class_to_number.values()]) == balanced_n, 'More entities than expected were saved'
        assert sum([v==smallest_entity for v in class_to_number.values()]) == balanced_n, 'Not all entities have the same number of images'
        assert target in class_to_number.keys()
    
        assert sum(len(files) for _, _, files in os.walk(dataset_forget_name)) == smallest_entity+1
        assert sum(len(files) for _, _, files in os.walk(dataset_retain_name)) == (balanced_n-1)*smallest_entity+1


        #if i > 5:
        #    break

# Each identity has about 15Mb
#!du -hs "{dataset_base_path_filtered}/{target}"

#!find "{dataset_forget_name}" \( -type f -o -type l \) | wc -l
#!find "{dataset_retain_name}"  \( -type f -o -type l \) | wc -l

In [ ]:
##############################
# Step 5: similarity matrix
##############################
# TODO: move to standalone **Notebook 2: Data Exploration**
#!rm "assets/similarity_clip_breeds.json"
df_similarities_clip = calculate_similarity_clip('breeds', restrict_labels)
plot_heatmap(df_similarities_clip)
df_similarities_clip.head()